# Base de Máquinas por MARCA — Formato de Fechamento

Converte as abas **`CC_Maquinas_2026`** (fonte principal de MARCA, coluna AK) e **`entre_empresas_abril`** (valores RSE e máquinas complementares) para o layout de `Base_formato_fechamento`.

## Como usar
1. Mantenha `CC_Maquinas_2026` atualizada no xlsx de entrada.
2. Opcional: revise `02-Referencias/Meus_Dados/de_para_marcas_maquinas.csv` para máquinas que existem só em `entre_empresas_abril`.
3. `data_nf` / `data_pagamento`: 2026 → `FECHAMENTO GERAL 2026 (HUGO).xlsx`; 2025 → `FECHAMENTO GERAL 2025.CSV`.
4. Aba **`base_maquinas_2025`**: catálogo `Carga_Ativos (CC Maquinas 2025).xlsx` (aba `2025`).
5. Execute todas as células.
6. Saída: `02-Referencias/relatorio_marcas_maquinas_base.xlsx` (abas 2026 + 2025).

In [7]:
from pathlib import Path
import re

import pandas as pd

REFS_DIR = Path('..') / '02-Referencias'
XLSX_IN = REFS_DIR / 'relatorio_marcas_maquinas.xlsx'
SHEET_CC = 'CC_Maquinas_2026'
SHEET_EE = 'entre_empresas_abril'
SHEET_MODELO = 'Base_formato_fechamento'
DEPARA_PATH = REFS_DIR / 'Meus_Dados' / 'de_para_marcas_maquinas.csv'
ODBC_PATH = REFS_DIR / 'Meus_Dados' / 'base.csv'
SAGI_CC_PATH = REFS_DIR / 'sagi_rel_centro_custo.csv'
def resolver_fechamento_2026_path() -> Path | None:
    candidatos = [
        REFS_DIR / 'Meus_Dados' / 'FECHAMENTO GERAL 2026 (HUGO).xlsx',
        REFS_DIR / 'FECHAMENTO GERAL 2026 (HUGO).xlsx',
        REFS_DIR / 'FECHAMENTO GERAL 2026 (HUGO).csv',
        REFS_DIR / 'Meus_Dados' / 'FECHAMENTO GERAL 2026 (HUGO).csv',
    ]
    for path in candidatos:
        if path.exists() and path.stat().st_size > 0:
            return path
    return None


FECHAMENTO_HUGO_PATH = resolver_fechamento_2026_path()
HUGO_SHEET_BASE = 'Base'
MES_REFERENCIA_DATA_NF = (4, 2026)  # prioriza lançamentos do mês (ex.: abril/2026)

CC_MAQUINAS_2025_PATH = REFS_DIR / 'Carga_Ativos (CC Maquinas 2025).xlsx'
SHEET_CC_2025 = '2025'
FECHAMENTO_2025_CSV_PATH = REFS_DIR / 'Meus_Dados' / 'FECHAMENTO GERAL 2025.CSV'
ANO_REFERENCIA_DATA_NF_2025 = 2025

XLSX_OUT = REFS_DIR / 'relatorio_marcas_maquinas_base.xlsx'

CONTA_LOCACAO = ('5.6.1', 'LOCACAO DE MAQUINAS E EQUIPAMENTOS')
CONTA_SERVICO = ('7.1.3', 'LOCACAO DE EQUIPAMENTOS E FERRAMENTAS')
CREDOR_COD = 362
CREDOR_NOME = 'RSE COMERCIO DE FERRO ACO DE EQUIPAMENTOS LTDA'

for path in (XLSX_IN, DEPARA_PATH, ODBC_PATH, SAGI_CC_PATH, CC_MAQUINAS_2025_PATH, FECHAMENTO_2025_CSV_PATH):
    if not path.exists():
        raise FileNotFoundError(path.resolve())

print('Arquivos carregados:')
print(f'  entrada: {XLSX_IN.resolve()}')
print(f'  abas:    {SHEET_CC}, {SHEET_EE}, {SHEET_MODELO}')
print(f'  de-para: {DEPARA_PATH.resolve()}')
print(f'  odbc:    {ODBC_PATH.resolve()}')
print(f'  sagi cc: {SAGI_CC_PATH.resolve()}')
if FECHAMENTO_HUGO_PATH is not None:
    print(f'  hugo 26: {FECHAMENTO_HUGO_PATH.resolve()}')
else:
    print('  hugo 26: (não encontrado — data_nf 2026 ficará vazia)')
print(f'  cc 2025: {CC_MAQUINAS_2025_PATH.resolve()} ({SHEET_CC_2025})')
print(f'  fech 25: {FECHAMENTO_2025_CSV_PATH.resolve()}')

Arquivos carregados:
  entrada: C:\Users\julio.santana\Documents\Projects\Cofre_Trabalho\02-Referencias\relatorio_marcas_maquinas.xlsx
  abas:    CC_Maquinas_2026, entre_empresas_abril, Base_formato_fechamento
  de-para: C:\Users\julio.santana\Documents\Projects\Cofre_Trabalho\02-Referencias\Meus_Dados\de_para_marcas_maquinas.csv
  odbc:    C:\Users\julio.santana\Documents\Projects\Cofre_Trabalho\02-Referencias\Meus_Dados\base.csv
  sagi cc: C:\Users\julio.santana\Documents\Projects\Cofre_Trabalho\02-Referencias\sagi_rel_centro_custo.csv
  hugo 26: C:\Users\julio.santana\Documents\Projects\Cofre_Trabalho\02-Referencias\FECHAMENTO GERAL 2026 (HUGO).xlsx
  cc 2025: C:\Users\julio.santana\Documents\Projects\Cofre_Trabalho\02-Referencias\Carga_Ativos (CC Maquinas 2025).xlsx (2025)
  fech 25: C:\Users\julio.santana\Documents\Projects\Cofre_Trabalho\02-Referencias\Meus_Dados\FECHAMENTO GERAL 2025.CSV


In [8]:
import unicodedata


def split_descen(descen: str) -> tuple[str, str, str, str, str]:
    partes = [p.strip() for p in str(descen).split('/') if p.strip()]
    natureza = partes[0] if partes else ''
    divisao = partes[1] if len(partes) > 1 else ''
    filial_cc = partes[2] if len(partes) > 2 else ''
    resto = partes[3:] if len(partes) > 3 else []
    n3 = resto[0] if resto else filial_cc
    n4 = ' / '.join(resto) if resto else n3
    if n3 and isinstance(n4, str) and n4.startswith(n3 + ' / '):
        n4 = n4[len(n3) + 3:].strip()
    if isinstance(n4, str) and ' / ' in n4:
        tail = n4.split(' / ')[-1].strip()
        if re.match(
            r'^([A-Z]{3}\d{4}|[A-Z]{3}\d[A-Z0-9]{3}|[A-Z0-9]{5,8}\s*\([^)]+\))$',
            tail,
            re.IGNORECASE,
        ):
            n4 = tail
    return natureza, divisao, filial_cc, n3, n4


def levels_from_codcen(codcen: str) -> tuple[str, str, str, str]:
    tokens = [t for t in str(codcen).strip().split('.') if t]
    if len(tokens) < 2:
        n1 = str(codcen).strip()
    else:
        n1 = '.'.join(tokens[:2])
    n2 = '.'.join(tokens[:3]) if len(tokens) >= 3 else n1
    n3 = '.'.join(tokens[:4]) if len(tokens) >= 4 else n2
    n4 = '.'.join(tokens) if tokens else ''
    return n1, n2, n3, n4


def placa_tokens(placa: str) -> list[str]:
    parts = re.split(r'[/\s]+', str(placa).strip())
    return [x for x in parts if x and re.match(r'^[A-Z0-9]', x, re.I)]


def normalizar_placa(placa: str) -> str:
    token = placa_tokens(placa)
    return token[0].upper() if token else str(placa).strip().upper()


def is_placa_valida(placa: str) -> bool:
    s = str(placa).strip().upper()
    if not s or s in {'-', 'NAN', '?'}:
        return False
    if 'UNIDADES' in s or s in {'GRIMAL'}:
        return False
    return bool(re.match(r'^[A-Z0-9]{3,}', s))


def is_tipo_maquina(tipo: str) -> bool:
    """Inclui MAQUINAS, PRENSA (móvel/fixa) e BALANÇA do CC_Maquinas_2026."""
    norm = unicodedata.normalize('NFKD', str(tipo or '')).encode('ascii', 'ignore').decode('ascii').upper()
    return any(k in norm for k in ('MAQUINA', 'PRENSA', 'BALANCA'))


def chave_placa(placa: str) -> str:
    return re.sub(r'\s+', ' ', str(placa).strip().upper())


def limpar_manual(val) -> str:
    if pd.isna(val):
        return ''
    txt = str(val).strip()
    return '' if txt.lower() in {'', 'nan', '-', '?'} else txt


def limpar_texto(valor) -> str:
    return limpar_manual(valor)


def map_localizacao_filial(localizacao: str) -> str:
    loc = str(localizacao or '').upper()
    mapping = [
        (r'CAMPO GRANDE', 'G3S CAMPO GRANDE'),
        (r'DOURADOS', 'G3S DOURADOS'),
        (r'LONDRINA', 'G3S LONDRINA'),
        (r'CIDADE ALTA', 'G3S MARINGA'),
        (r'MARING', 'G3S MARINGA'),
        (r'PRUDENTE', 'G3S PRUDENTE'),
        (r'BARRA MANSA', 'G&S BARUERI'),
        (r'RESENDE', 'G&S PRUDENTE'),
        (r'JOINVILLE', 'G&S BARUERI'),
        (r'PIRACICABA', 'G&S PRUDENTE'),
        (r'^RSE$', 'RSE'),
    ]
    for pattern, filial in mapping:
        if re.search(pattern, loc):
            return filial
    return ''


def infer_marca_auto(placa: str, implemento: str) -> tuple[str, str]:
    impl = str(implemento or '').upper()
    plc = str(placa or '').upper()
    if re.search(r'LIEBHERR|LH30M', impl) or plc.startswith('EHL'):
        return 'LIEBHERR', 'regra_automatica'
    if 'JCG800' in impl or 'JCG600' in impl:
        return 'JCG', 'regra_automatica'
    # JCH200M no IMPLEMENTO é modelo da prensa, não fabricante (ex.: HIDROEUROPA)
    if re.search(r'PRENSA\s+(30X30|40X40|60X60)', impl):
        return 'PENDENTE_VALIDACAO', 'regra_prensa_sem_modelo'
    if plc.startswith('BAL'):
        return 'PENDENTE_VALIDACAO', 'regra_balanca'
    if plc.startswith('BHM'):
        return 'PENDENTE_VALIDACAO', 'regra_tilter'
    if plc.startswith('EHH'):
        return 'PENDENTE_VALIDACAO', 'regra_ehh_sem_fabricante'
    return 'PENDENTE_VALIDACAO', 'regra_fallback'


def buscar_depara(de_para: pd.DataFrame, placa: str, placa_norm: str) -> pd.Series | None:
    chaves = {str(placa).strip(), str(placa_norm).strip(), chave_placa(placa)}
    chaves |= set(placa_tokens(placa))
    hit = de_para[de_para['placa'].astype(str).str.strip().isin(chaves)]
    return hit.iloc[0] if len(hit) else None


def resolver_marca(
    placa: str,
    implemento: str,
    marca_cc: str,
    de_para: pd.DataFrame,
    placa_norm: str = '',
    fonte_cc_label: str = 'CC_Maquinas_2026',
) -> tuple[str, str]:
    marca_cc = limpar_texto(marca_cc)
    if marca_cc:
        return marca_cc, fonte_cc_label

    hit = buscar_depara(de_para, placa, placa_norm or normalizar_placa(placa))
    if hit is not None:
        return str(hit['marca']), str(hit['fonte'])

    return infer_marca_auto(placa, implemento)


def buscar_odbc(placa: str, odbc: pd.DataFrame) -> pd.Series | None:
    for code in placa_tokens(placa):
        for col in ('descen', 'observacao'):
            mask = odbc[col].astype(str).str.contains(re.escape(code), case=False, na=False)
            matches = odbc[mask]
            if len(matches):
                analytic = matches[matches['descen'].astype(str).str.contains(code, case=False, na=False)]
                if len(analytic):
                    return analytic.iloc[0]
                return matches.iloc[0]
    return None


def carregar_sagi_map(path: Path = SAGI_CC_PATH) -> dict[str, str]:
    sagi_map: dict[str, str] = {}
    skip_words = {'ANALITICO', 'SINTETICO', 'ANÁLTICO', 'SINTÉTICO'}
    for line in path.read_text(encoding='latin-1').splitlines():
        if ';' not in line:
            continue
        parts = [p.strip() for p in line.split(';')]
        codcen = parts[0]
        if not re.match(r'^\d', codcen):
            continue
        descricao = next((p for p in parts[1:] if p and p.upper() not in skip_words), '')
        if descricao:
            sagi_map[codcen] = descricao
    return sagi_map


def build_descen_from_sagi(codcen: str, sagi_map: dict[str, str]) -> str:
    tokens = [t for t in str(codcen).split('.') if t]
    labels: list[str] = []
    for i in range(1, len(tokens) + 1):
        code = '.'.join(tokens[:i])
        if code in sagi_map:
            labels.append(sagi_map[code])
    if not labels:
        return ''
    if labels[0].upper() != 'DESPESA':
        return 'DESPESA / ' + ' / '.join(labels)
    return ' / '.join(labels)


def buscar_sagi_codcen(placa: str, sagi_map: dict[str, str]) -> tuple[str, str] | None:
    candidatos: list[tuple[str, str]] = []
    for code in placa_tokens(placa):
        for codcen, descricao in sagi_map.items():
            if code.upper() in descricao.upper():
                candidatos.append((codcen, descricao))
    if not candidatos:
        return None
    candidatos.sort(key=lambda item: len(item[0]), reverse=True)
    return candidatos[0]


def resolver_centro_custo(
    placa: str,
    odbc: pd.DataFrame,
    sagi_map: dict[str, str],
    manual_codcen: str,
    manual_descen: str,
    localizacao: str,
) -> tuple[str, str, str, str]:
    if manual_codcen and manual_descen:
        return manual_codcen, manual_descen, map_localizacao_filial(localizacao), 'de_para_manual'

    odbc_row = buscar_odbc(placa, odbc)
    if odbc_row is not None:
        filial = str(odbc_row.get('filial', '') or '').strip()
        return str(odbc_row['codcen']), str(odbc_row['descen']), filial, 'odbc_placa'

    hit = buscar_sagi_codcen(placa, sagi_map)
    if hit is not None:
        codcen, _ = hit
        descen = build_descen_from_sagi(codcen, sagi_map)
        if descen:
            odbc_cod = odbc[odbc['codcen'].astype(str).str.strip() == codcen]
            filial = str(odbc_cod.iloc[0]['filial']).strip() if len(odbc_cod) else map_localizacao_filial(localizacao)
            fonte = 'odbc_codcen_sagi' if len(odbc_cod) else 'sagi_descen'
            return codcen, descen, filial, fonte

    return '', '', map_localizacao_filial(localizacao), 'faltante'


def montar_hierarquia_cc(codcen: str, descen: str, placa: str) -> dict[str, str]:
    _, divisao, filial_cc, n3_desc, n4_desc = split_descen(descen)
    n1_cod, n2_cod, n3_cod, n4_cod = levels_from_codcen(codcen)

    def label(code: str, desc: str) -> str:
        return f'{code} {desc}'.strip()

    n1_desc = divisao or filial_cc
    n2_desc = filial_cc or n3_desc
    if not n3_desc:
        n3_desc = n2_desc

    placa_txt = str(placa).strip()
    return {
        'n1_cod_centro_custo': n1_cod,
        'n1_centro_custo': n1_desc,
        'n1_CC': label(n1_cod, n1_desc),
        'n2_cod_centro_custo': n2_cod,
        'n2_centro_custo': n2_desc,
        'n2_CC': label(n2_cod, n2_desc),
        'n3_cod_centro_custo': n3_cod,
        'n3_centro_custo': n3_desc,
        'n3_CC': label(n3_cod, n3_desc),
        'n4_cod_centro_custo': n4_cod,
        'n4_centro_custo': placa_txt,
        'n4_CC': label(n4_cod, placa_txt),
    }


def to_float(valor) -> float | None:
    if pd.isna(valor):
        return None
    if isinstance(valor, (int, float)):
        return float(valor)
    txt = str(valor).strip().replace('R$', '').replace(' ', '')
    if not txt or txt == '-':
        return None
    txt = txt.replace('.', '').replace(',', '.') if ',' in txt else txt
    try:
        return float(txt)
    except ValueError:
        return None


def carregar_catalogo_cc(path: Path = XLSX_IN) -> pd.DataFrame:
    df = pd.read_excel(path, sheet_name=SHEET_CC)
    df = df[df['TIPO'].apply(is_tipo_maquina)].copy()
    df = df[df['PLACA'].apply(is_placa_valida)].copy()

    implemento = df['IMPLEMENTO'].fillna('').astype(str)
    modelo = df['MODELO'].fillna('').astype(str)
    df['implemento_txt'] = implemento.where(implemento.str.strip().ne(''), modelo)

    loc_fixo = 'Locação 0 Fixo 0 fev/2025'
    df['valor_deprec_cc'] = df['DEPRECIAÇÃO'].apply(to_float)
    if loc_fixo in df.columns:
        df['valor_locacao_cc'] = df[loc_fixo].apply(to_float)
    else:
        df['valor_locacao_cc'] = None

    df['placa_norm'] = df['PLACA'].apply(normalizar_placa)
    return df.reset_index(drop=True)


def carregar_catalogo_ee(path: Path = XLSX_IN) -> pd.DataFrame:
    df = pd.read_excel(path, sheet_name=SHEET_EE)
    df = df[df['TIPO'].apply(is_tipo_maquina)].copy()
    df['placa_norm'] = df['PLACA'].apply(normalizar_placa)
    df['valor_abril_ee'] = df.get('FATURA RSE ABRIL 2026 (1°)', pd.Series(dtype=float)).apply(to_float)
    df['valor_deprec_ee'] = df.get('VALOR DA LOCAÇÃO (DEPRECIAÇÃO)', pd.Series(dtype=float)).apply(to_float)
    return df.reset_index(drop=True)


def carregar_catalogo_cc_2025(
    path: Path = CC_MAQUINAS_2025_PATH,
    sheet: str = SHEET_CC_2025,
) -> pd.DataFrame:
    df = pd.read_excel(path, sheet_name=sheet, header=1)
    df = df[df['TIPO'].apply(is_tipo_maquina)].copy()
    df = df[df['PLACA'].apply(is_placa_valida)].copy()

    implemento = df['IMPLEMENTO'].fillna('').astype(str)
    modelo = df['MODELO'].fillna('').astype(str)
    df['implemento_txt'] = implemento.where(implemento.str.strip().ne(''), modelo)

    loc_col = 'Locação - Fixo - fev/2025'
    df['valor_deprec_cc'] = df['DEPRECIAÇÃO'].apply(to_float) if 'DEPRECIAÇÃO' in df.columns else None
    if loc_col in df.columns:
        df['valor_locacao_cc'] = df[loc_col].apply(to_float)
    else:
        df['valor_locacao_cc'] = None

    df['placa_norm'] = df['PLACA'].apply(normalizar_placa)
    return df.reset_index(drop=True)


def montar_catalogo_from_cc(df: pd.DataFrame, fonte: str) -> pd.DataFrame:
    linhas = []
    for _, r in df.iterrows():
        linhas.append({
            'placa': r['PLACA'],
            'placa_norm': r['placa_norm'],
            'implemento': limpar_texto(r.get('implemento_txt')),
            'localizacao': limpar_texto(r.get('LOCALIZAÇÃO')),
            'marca_cc': limpar_texto(r.get('MARCA')),
            'modelo': limpar_texto(r.get('MODELO')),
            'locacao_servico': 'LOCAÇÃO',
            'valor_abril_ee': None,
            'valor_deprec_ee': None,
            'valor_deprec_cc': r.get('valor_deprec_cc'),
            'valor_locacao_cc': r.get('valor_locacao_cc'),
            'obs_controladoria': '',
            'fonte_dados': fonte,
        })
    return pd.DataFrame(linhas)


def montar_catalogo_unificado(cc: pd.DataFrame, ee: pd.DataFrame) -> pd.DataFrame:
    """Une CC e entre_empresas por placa_norm; dados do CC (incl. MARCA) têm prioridade."""
    cc_idx = {r['placa_norm']: r for _, r in cc.iterrows()}
    ee_idx = {r['placa_norm']: r for _, r in ee.iterrows()}
    chaves = sorted(set(cc_idx) | set(ee_idx))
    linhas = []

    for chave in chaves:
        cc_data = cc_idx.get(chave)
        ee_data = ee_idx.get(chave)

        placa = (cc_data['PLACA'] if cc_data is not None else ee_data['PLACA'])
        implemento = ''
        if cc_data is not None:
            implemento = limpar_texto(cc_data.get('implemento_txt'))
        if not implemento and ee_data is not None:
            implemento = limpar_texto(ee_data.get('IMPLEMENTO'))

        localizacao = ''
        if cc_data is not None:
            localizacao = limpar_texto(cc_data.get('LOCALIZAÇÃO'))
        if not localizacao and ee_data is not None:
            localizacao = limpar_texto(ee_data.get('LOCALIZAÇÃO'))

        locacao_servico = ''
        if ee_data is not None:
            locacao_servico = limpar_texto(ee_data.get('LOCAÇÃO OU SERVIÇO'))
        if not locacao_servico:
            locacao_servico = 'LOCAÇÃO'

        obs_ctrl = ''
        if ee_data is not None and pd.notna(ee_data.get('OBSERVAÇÕES CONTROLADORIA')):
            obs_ctrl = limpar_texto(ee_data.get('OBSERVAÇÕES CONTROLADORIA'))

        fontes = []
        if cc_data is not None:
            fontes.append('CC_Maquinas_2026')
        if ee_data is not None:
            fontes.append('entre_empresas_abril')

        linhas.append({
            'placa': placa,
            'placa_norm': chave,
            'implemento': implemento,
            'localizacao': localizacao,
            'marca_cc': limpar_texto(cc_data['MARCA']) if cc_data is not None else '',
            'modelo': limpar_texto(cc_data['MODELO']) if cc_data is not None else '',
            'locacao_servico': locacao_servico,
            'valor_abril_ee': ee_data['valor_abril_ee'] if ee_data is not None else None,
            'valor_deprec_ee': ee_data['valor_deprec_ee'] if ee_data is not None else None,
            'valor_deprec_cc': cc_data['valor_deprec_cc'] if cc_data is not None else None,
            'valor_locacao_cc': cc_data['valor_locacao_cc'] if cc_data is not None else None,
            'obs_controladoria': obs_ctrl,
            'fonte_dados': '+'.join(fontes),
        })

    return pd.DataFrame(linhas)


def valor_referencia_catalogo(row: pd.Series) -> float | None:
    for col in ('valor_abril_ee', 'valor_deprec_ee', 'valor_deprec_cc', 'valor_locacao_cc'):
        val = row.get(col)
        if pd.notna(val) and float(val) > 0:
            return float(val)
    return None


def fmt_data_nf(v) -> str:
    if pd.isna(v) or v == '':
        return ''
    dt = pd.to_datetime(v, errors='coerce')
    if pd.isna(dt):
        return ''
    return dt.strftime('%d/%m/%Y')


def extrair_placa_n4_hugo(n4) -> str:
    if pd.isna(n4):
        return ''
    s = str(n4).strip().upper()
    if re.match(r'^[A-Z]{3}\d', s) or re.match(r'^[A-Z]{3}[A-Z0-9]{4,}', s):
        return re.split(r'[\s(/]', s)[0]
    return ''


def score_linha_hugo(
    row: pd.Series,
    mes_ref: tuple[int, int] | int | None = MES_REFERENCIA_DATA_NF,
) -> int:
    score = 0
    credor = str(row.get('credor_forn_cli_func', '')).upper()
    if 'RSE COMERCIO' in credor:
        score += 100
    cod = str(row.get('cod_conta', ''))
    if cod.startswith('5.6.1'):
        score += 50
    elif cod.startswith('7.1.3'):
        score += 40
    conta = str(row.get('conta', '')).upper()
    if 'LOCACAO' in conta or 'LOCAÇÃO' in conta:
        score += 30
    dt = pd.to_datetime(row.get('data_nf'), errors='coerce')
    if mes_ref is not None and pd.notna(dt):
        if isinstance(mes_ref, int) and dt.year == mes_ref:
            score += 20
        elif isinstance(mes_ref, tuple) and (dt.month, dt.year) == mes_ref:
            score += 20
    return score


def tokens_hugo_linha(row: pd.Series) -> list[str]:
    tokens: list[str] = []
    placa_n4 = extrair_placa_n4_hugo(row.get('n4_centro_custo'))
    if placa_n4:
        tokens.append(placa_n4)
    texto = ' '.join(str(row.get(c, '')) for c in ('n4_centro_custo', 'titulo', 'observacao'))
    for tok in placa_tokens(texto):
        if tok not in tokens:
            tokens.append(tok)
    return tokens


def carregar_mapa_data_nf_fechamento(
    path: Path,
    mes_ref: tuple[int, int] | int | None = MES_REFERENCIA_DATA_NF,
    sheet: str | None = HUGO_SHEET_BASE,
) -> dict[str, dict]:
    """Índice placa/token -> melhor data_nf no fechamento (xlsx ou csv)."""
    cols = [
        'n4_centro_custo', 'titulo', 'observacao', 'data_nf', 'data_pagamento',
        'cod_conta', 'conta', 'credor_forn_cli_func',
    ]
    if path.suffix.lower() == '.csv':
        df = pd.read_csv(path, sep=';', encoding='latin-1', usecols=cols, low_memory=False)
    else:
        df = pd.read_excel(path, sheet_name=sheet or HUGO_SHEET_BASE)
        df = df[[c for c in cols if c in df.columns]]

    df['data_nf'] = pd.to_datetime(df['data_nf'], errors='coerce')
    if 'data_pagamento' in df.columns:
        df['data_pagamento'] = pd.to_datetime(df['data_pagamento'], errors='coerce')
    df = df[df['data_nf'].notna()].copy()

    mapa: dict[str, dict] = {}
    for _, row in df.iterrows():
        score = score_linha_hugo(row, mes_ref)
        hit = {
            'data_nf': row['data_nf'],
            'data_pagamento': row.get('data_pagamento'),
            'score': score,
            'conta': str(row.get('conta', '')),
            'credor': str(row.get('credor_forn_cli_func', '')),
        }
        for tok in tokens_hugo_linha(row):
            atual = mapa.get(tok)
            if atual is None or hit['score'] > atual['score'] or (
                hit['score'] == atual['score'] and hit['data_nf'] > atual['data_nf']
            ):
                mapa[tok] = hit
    return mapa


def carregar_mapa_data_nf_hugo(
    path: Path | None = None,
    sheet: str = HUGO_SHEET_BASE,
    mes_ref: tuple[int, int] | None = MES_REFERENCIA_DATA_NF,
) -> dict[str, dict]:
    alvo = path or FECHAMENTO_HUGO_PATH
    if alvo is None:
        return {}
    return carregar_mapa_data_nf_fechamento(alvo, mes_ref=mes_ref, sheet=sheet)


def montar_base_formato_fechamento(
    catalogo: pd.DataFrame,
    modelo_cols: list[str],
    mapa_data_nf: dict[str, dict],
    de_para: pd.DataFrame,
    odbc: pd.DataFrame,
    sagi_map: dict[str, str],
    fonte_data_nf_label: str = 'FECHAMENTO_HUGO',
) -> tuple[pd.DataFrame, pd.DataFrame]:
    out_rows: list[dict] = []
    diag_rows: list[dict] = []
    col_divisao = 'Divisao' if 'Divisao' in modelo_cols else 'sdssds'

    for _, row in catalogo.iterrows():
        placa = row['placa']
        placa_norm = row['placa_norm']
        implemento = row['implemento']
        fonte_cc_label = str(row.get('fonte_dados', 'CC_Maquinas_2026')).split('+')[0].strip()
        marca, fonte_marca = resolver_marca(
            placa, implemento, row['marca_cc'], de_para, placa_norm, fonte_cc_label=fonte_cc_label
        )

        depara_row = buscar_depara(de_para, placa, placa_norm)
        manual_codcen = limpar_manual(depara_row.get('codcen_manual')) if depara_row is not None else ''
        manual_descen = limpar_manual(depara_row.get('descen_manual')) if depara_row is not None else ''

        codcen, descen, filial, fonte_cc = resolver_centro_custo(
            placa, odbc, sagi_map, manual_codcen, manual_descen, row['localizacao']
        )
        if codcen and descen:
            cc = montar_hierarquia_cc(codcen, descen, placa)
        else:
            cc = {k: '' for k in [
                'n1_cod_centro_custo', 'n1_centro_custo', 'n1_CC',
                'n2_cod_centro_custo', 'n2_centro_custo', 'n2_CC',
                'n3_cod_centro_custo', 'n3_centro_custo', 'n3_CC',
                'n4_cod_centro_custo', 'n4_centro_custo', 'n4_CC',
            ]}
            cc['n4_centro_custo'] = str(placa).strip()
            cc['n4_CC'] = str(placa).strip()

        if not filial:
            filial = map_localizacao_filial(row['localizacao'])

        locacao_servico = str(row.get('locacao_servico', 'LOCAÇÃO')).strip().upper()
        cod_conta, conta = CONTA_SERVICO if 'SERV' in locacao_servico else CONTA_LOCACAO

        valor = valor_referencia_catalogo(row)
        obs_parts = [implemento]
        if row.get('modelo'):
            obs_parts.append(f"modelo={row['modelo']}")
        if row.get('obs_controladoria'):
            obs_parts.append(row['obs_controladoria'])
        observacao = ' | '.join(p for p in obs_parts if p)

        hit_data = buscar_data_nf_hugo(placa, placa_norm, mapa_data_nf)

        registro = {c: '' for c in modelo_cols}
        registro.update(cc)
        registro[col_divisao] = cc.get('n1_centro_custo', '')
        registro['MARCA'] = marca
        registro['cod_conta'] = cod_conta
        registro['conta'] = conta
        registro['cod_conta-descr'] = f'{cod_conta} {conta}'
        registro['filial'] = filial
        registro['titulo'] = placa_tokens(placa)[0] if placa_tokens(placa) else str(placa)
        if valor is not None:
            registro['valor_nf'] = valor
            registro['valor_conta'] = valor
            registro['Valor Oficial'] = valor
        registro['observacao'] = observacao
        if hit_data:
            registro['data_nf'] = fmt_data_nf(hit_data.get('data_nf'))
            pag = hit_data.get('data_pagamento')
            if pd.notna(pag):
                registro['data_pagamento'] = fmt_data_nf(pag)
        registro['cod_credor_forn_cli_func'] = CREDOR_COD
        registro['credor_forn_cli_func'] = CREDOR_NOME
        registro['Origem'] = row['fonte_dados']
        registro['Sistema'] = 'Controladoria'
        aux = [f'placa={placa}', f'fonte_marca={fonte_marca}', f'fonte_cc={fonte_cc}']
        if hit_data:
            aux.append(f'fonte_data_nf={fonte_data_nf_label}')
            if hit_data.get('conta'):
                aux.append(f"conta_fechamento={hit_data['conta']}")
        registro['Dados auxiliares'] = ';'.join(aux)
        out_rows.append(registro)

        diag_rows.append({
            'PLACA': placa,
            'fonte_dados': row['fonte_dados'],
            'IMPLEMENTO': implemento,
            'MARCA': marca,
            'fonte_marca': fonte_marca,
            'fonte_cc': fonte_cc,
            'CC_resolvido': 'OK' if fonte_cc != 'faltante' else 'FALTANTE',
            'filial_localizacao': filial or map_localizacao_filial(row['localizacao']),
            'valor_referencia': valor,
            'data_nf': registro.get('data_nf', ''),
            'fonte_data_nf': fonte_data_nf_label if hit_data else '',
            'pendente_validacao': marca == 'PENDENTE_VALIDACAO',
        })

    return pd.DataFrame(out_rows, columns=modelo_cols), pd.DataFrame(diag_rows)


def buscar_data_nf_hugo(placa: str, placa_norm: str, mapa: dict[str, dict]) -> dict:
    chaves = list(dict.fromkeys([*placa_tokens(placa), str(placa_norm).strip()]))
    for tok in chaves:
        if tok and tok in mapa:
            return mapa[tok]
    return {}

In [9]:
modelo_cols = pd.read_excel(XLSX_IN, sheet_name=SHEET_MODELO, nrows=0).columns.tolist()
de_para = pd.read_csv(DEPARA_PATH, sep=';', encoding='utf-8-sig')
odbc = pd.read_csv(ODBC_PATH, sep=';', encoding='latin-1', low_memory=False)
sagi_map = carregar_sagi_map()

cc_raw = carregar_catalogo_cc()
ee_raw = carregar_catalogo_ee()
catalogo = montar_catalogo_unificado(cc_raw, ee_raw)
if FECHAMENTO_HUGO_PATH is not None:
    mapa_data_nf_hugo = carregar_mapa_data_nf_hugo(FECHAMENTO_HUGO_PATH)
else:
    mapa_data_nf_hugo = {}
    print('Aviso: fechamento 2026 não encontrado; data_nf da base 2026 ficará vazia.')

print(f'CC_Maquinas_2026 (máquinas válidas): {len(cc_raw)}')
print(f'Fechamento 2026 data_nf (tokens):    {len(mapa_data_nf_hugo)}')
print(f'entre_empresas_abril (máquinas):     {len(ee_raw)}')
print(f'Catálogo unificado (sem duplicar):   {len(catalogo)}')
print(f'Colunas do modelo de fechamento:     {len(modelo_cols)}')

print('Catálogo 2026 pronto; base e diagnóstico serão montados na próxima célula.')

CC_Maquinas_2026 (máquinas válidas): 71
Fechamento 2026 data_nf (tokens):    20410
entre_empresas_abril (máquinas):     41
Catálogo unificado (sem duplicar):   73
Colunas do modelo de fechamento:     36
Catálogo 2026 pronto; base e diagnóstico serão montados na próxima célula.


In [10]:
base_maquinas, diagnostico = montar_base_formato_fechamento(
    catalogo,
    modelo_cols,
    mapa_data_nf_hugo,
    de_para,
    odbc,
    sagi_map,
    fonte_data_nf_label='FECHAMENTO_HUGO_2026',
)

col_divisao = 'Divisao' if 'Divisao' in modelo_cols else 'sdssds'
preview_cols = [c for c in [col_divisao, 'n4_centro_custo', 'MARCA', 'filial', 'cod_conta', 'valor_nf', 'data_nf', 'Origem'] if c in base_maquinas.columns]

print('\n--- Cobertura 2026 ---')
print(f"MARCA preenchida: {diagnostico['MARCA'].notna().sum()}/{len(diagnostico)}")
print(f"MARCA via CC_Maquinas_2026: {(diagnostico['fonte_marca'] == 'CC_Maquinas_2026').sum()}")
print(f"PENDENTE_VALIDACAO: {diagnostico['pendente_validacao'].sum()}")
print(f"CC resolvido: {(diagnostico['CC_resolvido'] == 'OK').sum()}/{len(diagnostico)}")
print(f"Valor referência: {diagnostico['valor_referencia'].notna().sum()}/{len(diagnostico)}")
print(f"data_nf:          {(diagnostico['data_nf'].astype(str).str.strip() != '').sum()}/{len(diagnostico)}")

print(f'\nBase 2026: {len(base_maquinas)} linhas x {len(base_maquinas.columns)} colunas')
display(base_maquinas[preview_cols].head(12))


--- Cobertura 2026 ---
MARCA preenchida: 73/73
MARCA via CC_Maquinas_2026: 71
PENDENTE_VALIDACAO: 2
CC resolvido: 53/73
Valor referência: 58/73
data_nf:          38/73

Base 2026: 73 linhas x 36 colunas


,Divisao,n4_centro_custo,MARCA,filial,cod_conta,valor_nf,data_nf,Origem
0,SELETIVA,BAL0001,CAPITAL,G3S MARINGA,5.6.1,2083.333333,,CC_Maquinas_2026+entre_empresas_abril
1,,BAL0002,CAPITAL,,5.6.1,5166.666667,,CC_Maquinas_2026
2,SELETIVA,BAL0003,CAPITAL,G3S LONDRINA,5.6.1,2083.333333,,CC_Maquinas_2026+entre_empresas_abril
3,EKIPA LOCACOES RSE,BAL0004,CAPITAL,G3S PRUDENTE,5.6.1,2083.333333,,CC_Maquinas_2026+entre_empresas_abril
4,,BAL0005,CAPITAL,,5.6.1,5166.666667,,CC_Maquinas_2026
5,SELETIVA,BAL0006,PADRÃO,G3S DOURADOS,5.6.1,2083.333333,,CC_Maquinas_2026+entre_empresas_abril
6,SELETIVA,BAL0007,PADRÃO,G3S CAMPO GRANDE,5.6.1,2083.333333,,CC_Maquinas_2026+entre_empresas_abril
7,,BAL0008,TOLEDO,,5.6.1,3333.333333,,CC_Maquinas_2026
8,,BAL0009,TOLEDO,,5.6.1,3333.333333,,CC_Maquinas_2026
9,,BAL0010,TOLEDO,,5.6.1,3333.333333,,CC_Maquinas_2026


In [11]:
print('Carregando catálogo 2025...')
cc_2025_raw = carregar_catalogo_cc_2025()
catalogo_2025 = montar_catalogo_from_cc(cc_2025_raw, fonte='CC_Maquinas_2025')

print(f'CC_Maquinas_2025 (máquinas válidas): {len(cc_2025_raw)}')
print('Indexando datas no FECHAMENTO GERAL 2025.CSV (pode levar alguns segundos)...')
mapa_data_nf_2025 = carregar_mapa_data_nf_fechamento(
    FECHAMENTO_2025_CSV_PATH,
    mes_ref=ANO_REFERENCIA_DATA_NF_2025,
    sheet=None,
)
print(f'Fechamento 2025 data_nf (tokens): {len(mapa_data_nf_2025)}')

base_maquinas_2025, diagnostico_2025 = montar_base_formato_fechamento(
    catalogo_2025,
    modelo_cols,
    mapa_data_nf_2025,
    de_para,
    odbc,
    sagi_map,
    fonte_data_nf_label='FECHAMENTO_2025_CSV',
)

preview_2025 = [c for c in [col_divisao, 'n4_centro_custo', 'MARCA', 'filial', 'valor_nf', 'data_nf', 'Origem'] if c in base_maquinas_2025.columns]
print('\n--- Cobertura 2025 ---')
print(f"MARCA via CC:       {(diagnostico_2025['fonte_marca'] == 'CC_Maquinas_2025').sum()}/{len(diagnostico_2025)}")
print(f"CC resolvido:       {(diagnostico_2025['CC_resolvido'] == 'OK').sum()}/{len(diagnostico_2025)}")
print(f"Valor referência:   {diagnostico_2025['valor_referencia'].notna().sum()}/{len(diagnostico_2025)}")
print(f"data_nf:            {(diagnostico_2025['data_nf'].astype(str).str.strip() != '').sum()}/{len(diagnostico_2025)}")
print(f'\nBase 2025: {len(base_maquinas_2025)} linhas')
display(base_maquinas_2025[preview_2025].head(12))

Carregando catálogo 2025...
CC_Maquinas_2025 (máquinas válidas): 67
Indexando datas no FECHAMENTO GERAL 2025.CSV (pode levar alguns segundos)...


C:\Users\julio.santana\AppData\Local\Temp\ipykernel_20632\2847557966.py:482: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['data_nf'] = pd.to_datetime(df['data_nf'], errors='coerce')
C:\Users\julio.santana\AppData\Local\Temp\ipykernel_20632\2847557966.py:484: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['data_pagamento'] = pd.to_datetime(df['data_pagamento'], errors='coerce')


Fechamento 2025 data_nf (tokens): 44020

--- Cobertura 2025 ---
MARCA via CC:       67/67
CC resolvido:       49/67
Valor referência:   36/67
data_nf:            36/67

Base 2025: 67 linhas


,Divisao,n4_centro_custo,MARCA,filial,valor_nf,data_nf,Origem
0,EKIPA LOCACOES RSE,EHL0012,LIEBHERR,G&S PRUDENTE,700.0,,CC_Maquinas_2025
1,EKIPA LOCACOES RSE,EHL0011,LIEBHERR,G&S PRUDENTE,,,CC_Maquinas_2025
2,EKIPA LOCACOES E SERV G&S,EHH0036,HYUNDAI,G&S BARUERI,9583.333333,31/12/2025,CC_Maquinas_2025
3,EKIPA LOCACOES E SERV G&S,EHH0037,HYUNDAI,G&S BARUERI,9583.333333,05/12/2025,CC_Maquinas_2025
4,EKIPA LOCACOES E SERV G&S,EHH0038,HYUNDAI,G&S BARUERI,9583.333333,31/12/2025,CC_Maquinas_2025
5,EKIPA LOCACOES E SERV G&S,EHH0040,HYUNDAI,G&S BARUERI,9583.333333,31/12/2025,CC_Maquinas_2025
6,EKIPA LOCACOES E SERV G&S,EHH0039,HYUNDAI,G&S BARUERI,9583.333333,31/12/2025,CC_Maquinas_2025
7,EKIPA LOCACOES E SERV G&S,EHH0002,HYUNDAI,G&S BARUERI,4916.666667,24/10/2025,CC_Maquinas_2025
8,EKIPA LOCACOES E SERV G&S,EHH0004,HYUNDAI,G&S BARUERI,4916.666667,31/12/2025,CC_Maquinas_2025
9,SELETIVA,EHH0005,HYUNDAI,G&S PRUDENTE,5166.666667,02/12/2025,CC_Maquinas_2025


In [12]:
assert len(base_maquinas) == len(catalogo), 'Base 2026 diverge do catálogo unificado'
assert len(base_maquinas_2025) == len(catalogo_2025), 'Base 2025 diverge do catálogo 2025'
assert base_maquinas['MARCA'].notna().all(), 'Existem linhas sem MARCA'
assert base_maquinas['n4_centro_custo'].astype(str).str.strip().ne('').all(), 'Existem linhas sem identificador de máquina'

cc_faltante = diagnostico.loc[diagnostico['CC_resolvido'] == 'FALTANTE']
if len(cc_faltante):
    print(f'Atenção: {len(cc_faltante)} PLACAs sem CC (ODBC/SAGI/manual):')
    display(cc_faltante[['PLACA', 'fonte_dados', 'filial_localizacao']])

pendentes = base_maquinas.loc[base_maquinas['MARCA'] == 'PENDENTE_VALIDACAO', ['n4_centro_custo', 'MARCA', 'observacao', 'Origem']]
print(f'Pendências de validação: {len(pendentes)}')
if len(pendentes):
    display(pendentes)

with pd.ExcelWriter(XLSX_OUT, engine='openpyxl') as writer:
    base_maquinas.to_excel(writer, sheet_name='base_maquinas_2026', index=False)
    base_maquinas_2025.to_excel(writer, sheet_name='base_maquinas_2025', index=False)
    diagnostico.to_excel(writer, sheet_name='diagnostico_2026', index=False)
    diagnostico_2025.to_excel(writer, sheet_name='diagnostico_2025', index=False)
    catalogo.to_excel(writer, sheet_name='catalogo_unificado_2026', index=False)
    catalogo_2025.to_excel(writer, sheet_name='catalogo_2025', index=False)

print(f'\nResumo final: 2026={len(base_maquinas)} | 2025={len(base_maquinas_2025)} máquinas')
print(f'Arquivo: {XLSX_OUT.resolve()}')

Atenção: 20 PLACAs sem CC (ODBC/SAGI/manual):


,PLACA,fonte_dados,filial_localizacao
1,BAL0002,CC_Maquinas_2026,
4,BAL0005,CC_Maquinas_2026,
7,BAL0008,CC_Maquinas_2026,
8,BAL0009,CC_Maquinas_2026,
9,BAL0010,CC_Maquinas_2026,
12,CHM0027,CC_Maquinas_2026,
40,EHN0001,CC_Maquinas_2026,
42,EPH0020,CC_Maquinas_2026,
48,GAR0001,CC_Maquinas_2026,G3S CAMPO GRANDE
51,IPM0061,CC_Maquinas_2026,


Pendências de validação: 2


,n4_centro_custo,MARCA,observacao,Origem
10,BHM0047,PENDENTE_VALIDACAO,TILTER | CONTRATO OK,entre_empresas_abril
29,EHH0044,PENDENTE_VALIDACAO,ESCAVADEIRA / TESOURA ARDEN | AGUARDAR - IRÁ E...,entre_empresas_abril



Resumo final: 2026=73 | 2025=67 máquinas
Arquivo: C:\Users\julio.santana\Documents\Projects\Cofre_Trabalho\02-Referencias\relatorio_marcas_maquinas_base.xlsx
